# Structure editing operations (Package A, Feature A4)

This notebook demonstrates the structure_ops API: sampling a configuration
from an average structure, repeating it into a supercell, substituting
species, checking the composition, and displacing atoms.  It uses the same
COD GaAs CIF fixture as [06_interop.ipynb](06_interop.ipynb) and the
structure_ops functions from `discuspdf`.

See [docs/STRUCTURE_OPS.md](../docs/STRUCTURE_OPS.md) for the full
documentation.

In [1]:
from pathlib import Path
import os
import sys
sys.path.insert(0, str(Path.cwd().parent))
os.chdir(Path.cwd().parent)

import numpy as np
from discuspdf import Lattice, Structure, repeat, substitute, displace, sample_configuration, check_composition
from discuspdf.cif_io import read_cif

## Read the COD GaAs CIF

In [2]:
cif = read_cif("tests/fixtures/interop/gaas_cod_9008845.cif")
s_avg = cif.structure
print(s_avg)
assert s_avg.model == "average"

Structure(n=8, formula=Ga4As4, model='average', supercell=(np.int64(1), np.int64(1), np.int64(1)), a=5.654 Å, b=5.654 Å, c=5.654 Å, alpha=90°, beta=90°, gamma=90°)


## Sample a configuration

In [3]:
s_conf = sample_configuration(s_avg, seed=42).structure
print(s_conf)
assert s_conf.model == "configuration"
assert s_conf.n_atoms == 8

Structure(n=8, formula=Ga4As4, model='configuration', supercell=(np.int64(1), np.int64(1), np.int64(1)), a=5.654 Å, b=5.654 Å, c=5.654 Å, alpha=90°, beta=90°, gamma=90°)


## Repeat 4Ã4Ã4

In [4]:
s_rep = repeat(s_conf, (4, 4, 4)).structure
print(s_rep)
assert s_rep.n_atoms == 512

Structure(n=512, formula=Ga256As256, model='configuration', supercell=(np.int64(4), np.int64(4), np.int64(4)), a=22.61 Å, b=22.61 Å, c=22.61 Å, alpha=90°, beta=90°, gamma=90°)


## Substitute 10% Ga â In

In [5]:
s_sub = substitute(s_rep, old="Ga", new="In", seed=42, fraction=0.10).structure
n_in = int(np.sum(s_sub.species == "In"))
print(f"In: {n_in}, Ga: {int(np.sum(s_sub.species == 'Ga'))}")
assert n_in == 26

In: 26, Ga: 230


## check_composition

In [6]:
check_composition(s_sub, {"Ga": 230, "In": 26, "As": 256})
print("OK")

OK


## Displace Ï=0.05 Ã

In [7]:
s_disp = displace(s_sub, seed=42, sigma=0.05).structure
print(s_disp)
assert s_disp.n_atoms == 512

Structure(n=512, formula=Ga230As256In26, model='configuration', supercell=(np.int64(4), np.int64(4), np.int64(4)), a=22.61 Å, b=22.61 Å, c=22.61 Å, alpha=90°, beta=90°, gamma=90°)


## Summary

* `sample_configuration` draws a disordered configuration from an average
  structure's occupancies (deterministic with `seed`).
* `repeat` builds a supercell (4Ã4Ã4 -> 512 atoms).
* `substitute` replaces a fraction of one species with another; the total
  composition is verified with `check_composition`.
* `displace` adds Gaussian displacements (Ï = 0.05 Ã) to every atom while
  preserving the atom count.